# 5. Statistiek van de hoogwaterstand, Texel Noordzee

Python-versie van het R-script **`statistics-wl10-omni-texeln-impr.Rmd`** (KNMI, februari 2026).

Omnidirectionele herhalingstijdwaarden van de hoogwaterstand voor Texel Noordzee (steunpunt Eierlandse Gat,
duinen), inclusief de tabellen:

1. vormparameter uit SEAS5-DCSM5 (roosterpunt p315);
2. locatie en schaal uit de metingen (`texno1.csv` t/m `texno4.csv`), met de schaal gecorrigeerd met
   1,15 / schaalfactor;
3. onzekerheid uit een bootstrap van de meetjaren, met modelfout;
4. Weibull-refit en tabellen (alleen omni);
5. een bootstrap van alleen de schaalschatting.

**Gebruik:** controleer de instellingen hieronder en voer daarna alle cellen uit. Dit notebook heeft de andere
notebooks niet nodig.

**Dezelfde resultaten als R:** de toevalsgetallen worden precies zo getrokken als in R (`set.seed(379233)`),
dus met dezelfde `nb` zijn de resultaten gelijk aan die van het R-script.

Volgend notebook: `6_production_wl.ipynb`.

## Instellingen
De standaardwaarden van de mappen en van `nb` staan in `kuststatistiek/settings.py`.

In [ ]:
from kuststatistiek import settings

input_path = settings.DATA_PATH  # map met de aangeleverde invoerdata
output_path = settings.OUTPUT_PATH  # map voor de resultaten
nb = settings.NB  # aantal bootstrap-trekkingen (R: nb <- 100)

# meetbestanden; R gebruikt texno1-4 (texno5.csv, 2017-2024, wordt in het R-script niet gebruikt)
texno_files = ["texno1.csv", "texno2.csv", "texno3.csv", "texno4.csv"]

## Voorbereiding
Vaste waarden uit het R-script en de gebruikte functies.

In [ ]:
import numpy as np
import pandas as pd
import pyreadr

from kuststatistiek.settings import Paths, ProductionBins, wl_settings, TEXEL_SEED
from kuststatistiek.rstream import texel_random_streams
from kuststatistiek.io.read_wl import read_wlmeastexeln
from kuststatistiek.gw import FitGW_iHilli, qGWdisp, collect
from kuststatistiek.production import weibull_refit_wl_omni
from kuststatistiek.rutils import save_pkl, r_apply_sd, r_sd, write_r_csv

MODELSTD = 0.1  # modelfout in de vormparameter van SEAS5
THETA0SD = 0.027  # schattingsfout in de vormparameter van SEAS5, overgenomen van het nabije Den Helder
SLR = 0.003 * (2035 - 2017)  # zeespiegelstijging tot 2035 [m]
SCALE_CORRECTION = 1.15

paths = Paths(input_path=input_path, output_path=output_path)
WL = wl_settings(nb=nb)  # constanten van het R-script
station = "texeln"
k = WL.kthresh - 1  # index van de gebruikte drempel (R: kthresh; Python telt vanaf 0)
md = WL.metadata(station)

## Stap 1 – SEAS5-DCSM5-data bij p315 (R-chunk `eiseas5`)
Uitvoer: `texeln_omni.pkl`

In [ ]:
wl = pyreadr.read_r(paths.input_path / "SEAS5_hwloc_230.RData")["hw"]["hw"].to_numpy()
N = len(wl)
save_pkl(paths.out(f"{station}_omni.pkl"), wl=wl, N=N)

## Stap 2 – Metingen (R-chunk `eimeas`)
Hoogwaterstanden uit de texno-bestanden (zonder trendcorrectie), plus een heel klein toevalsgetal zoals in R.

Uitvoer: `texeln_omni_meas_pot.pkl`

In [ ]:
a = read_wlmeastexeln(paths.input_path, texno_files)
dtm = a["t"]
Nm = len(a["wspeed"])
# toevalsgetallen van R voor deze stap en voor stap 3, 5 en 7 (begint zelf met set.seed(379233))
streams = texel_random_streams(TEXEL_SEED, Nm, len(pd.unique(dtm.year)), nb)
wlm = a["wspeed"] + (streams["eimeas"].runif(Nm) - 0.5) * 0.0001
save_pkl(
    paths.out(f"{station}_omni_meas_pot.pkl"),
    wlm=wlm,
    dtm=dtm.to_numpy(),
    Nm=Nm,
    hwtrend=a["hwtrend"],
)
print(f"{Nm} hoogwaters, van {dtm.min()} tot {dtm.max()}")

## Stap 3 – GW-staart uit SEAS5-DCSM5 (R-chunk `dir-es-seas5dcms5`)
Fit de GW-staart aan de SEAS5-DCSM5-data en trek `nb` waarden van de vormparameter (normale verdeling met
standaardafwijking `THETA0SD`) voor de bootstrap.

Uitvoer: `texeln_stats_omni_boot.pkl`

In [ ]:
l0 = np.ceil(N * WL.pthreshold).astype(np.int64)  # aantal waarden boven elke drempel
X = -np.sort(-wl)  # gesorteerd van hoog naar laag
esGW = FitGW_iHilli(
    X[: int(l0.max()) + 20], p=WL.p, N=N, r11=1, l0=l0, sigma=np.inf, metadata=md
)
esGW["freq"] = WL.freq
esGW["EI"] = WL.EIvalue
theta0bs = streams["shape"].rnorm(nb) * THETA0SD + esGW["tailindex"][k]
save_pkl(paths.out(f"{station}_stats_omni_boot.pkl"), esGW=esGW, theta0bs=theta0bs)

## Stap 4 – GW-staart uit de metingen (R-chunk `omni-estimates-meas`)
Fit de GW-staart aan de metingen met de vormparameter van SEAS5-DCSM5. Daarna wordt de schaal gecorrigeerd
met 1,15 / schaalfactor (schaalfactor = schaal uit de metingen / schaal uit SEAS5-DCSM5), in lijn met de
hoofdstations.

Uitvoer: `texeln_stats_omni_meas.pkl`, `texeln_stats_omni_meas0.pkl`, `texeln_scalefactors.csv`

In [ ]:
l0m = np.maximum(3, np.round(Nm * WL.pthreshold)).astype(np.int64)
wlms = -np.sort(-wlm)
esmeasGW = FitGW_iHilli(
    wlms[: int(l0m.max()) + 20],
    WL.p,
    N=Nm,
    r11=1,
    fixedpar={"theta0": esGW["tailindex"][k], "theta0Std": esGW["tailindexStd"][k]},
    l0=l0m,
    sigma=np.inf,
    metadata=md,
)
esmeasGW["freq"] = WL.freq
esmeasGW["EI"] = WL.EIvalue
scalefactor = esmeasGW["scale"][k] / esGW["scale"][k]
save_pkl(paths.out(f"{station}_stats_omni_meas.pkl"), esmeasGW=[esmeasGW])


def corrected_q(es, p_out):
    """Kwantielen van GW-fit `es` bij kansen p_out, met de schaal gecorrigeerd met 1,15/schaalfactor."""
    p = es["k"][k] / es["N"]
    return qGWdisp(
        p_out,
        p,
        es["location"][k],
        es["logdisp"][k] + np.log(SCALE_CORRECTION / scalefactor),
        es["tailindex"][k],
        es["locationStd"][k],
        es["logdispStd"][k],
        es["tailindexStd"][k],
    )


mum0 = mum = np.exp(-WL.x) * WL.factor
qmum = corrected_q(esmeasGW, mum / WL.factor)
save_pkl(
    paths.out(f"{station}_stats_omni_meas0.pkl"),
    mum=mum,
    mum0=mum0,
    qmum=qmum,
    freq=WL.freq,
)


def fmt(t):
    return pd.Timestamp(t).strftime("%Y-%m-%d %H:%M:%S")


write_r_csv(
    paths.out(f"{station}_scalefactors.csv"),
    [
        ["Texel Noordzee"],
        [round(Nm * WL.delt, 1)],
        [fmt(dtm.min())],
        [fmt(dtm.max())],
        [round(scalefactor, 2)],
    ],
    ["station", "length", "start", "stop", "scale factor"],
)
print(f"schaalfactor: {scalefactor:.3f}")

## Stap 5 – Bootstrap met modelfout (R-chunk `dir-estimate-meas-bootstrap-moderr`)
Trek `nb` bootstrap-steekproeven van de meetjaren. Voor elke steekproef is de vormparameter de waarde die in
stap 3 is getrokken plus een toevallige modelfout (`MODELSTD`), en wordt de GW-staart opnieuw gefit.

Uitvoer: `texeln_stats_omni_meas2.pkl`

In [ ]:
yr = dtm.year.to_numpy()
yrm = pd.unique(yr)  # de meetjaren
rng = streams["bootstrap"]
yrmbs = (
    rng.sample(yrm, nb * len(yrm)).reshape(nb, len(yrm)).T
)  # herbemonsterde jaren, één kolom per steekproef
qmbs = np.full((len(WL.freq), nb), np.nan)
for ib in range(nb):
    wlmbs = wlm[
        collect(yr, yrmbs[:, ib]) - 1
    ]  # de hoogwaters van de herbemonsterde jaren
    theta00bs = theta0bs[ib] + MODELSTD * rng.rnorm(1)[0]
    Xb = -np.sort(-wlmbs)
    es = FitGW_iHilli(
        Xb[: int(l0m.max()) + 20],
        WL.p,
        N=Nm,
        r11=1,
        fixedpar={"theta0": theta00bs, "theta0Std": 0.0},
        l0=l0m,
        sigma=np.inf,
        metadata=md,
    )
    qmbs[:, ib] = corrected_q(es, WL.freq / WL.factor)["quantile"]
qmsd = r_apply_sd(qmbs, 1)  # standaardafwijking per herhalingstijd
save_pkl(
    paths.out(f"{station}_stats_omni_meas2.pkl"),
    mum=mum,
    mum0=mum0,
    freq=WL.freq,
    qmum=qmum,
    qmsd=qmsd,
)

## Stap 6 – Weibull-refit en tabellen (R-chunk `wblfit`)
Uitvoer: de tabellen van Texel Noordzee (`texeln_2035_wl_...csv`)

In [ ]:
weibull_refit_wl_omni(station, mum0, qmum, qmsd, WL.freq, ProductionBins(), paths, SLR)

## Stap 7 – Bootstrap van alleen de schaalschatting (R-chunk *Bootstrap the scale estimates only*)
Dezelfde herbemonstering van jaren, met de vormparameter vast op de schatting uit SEAS5-DCSM5. Geeft de
standaardafwijking van de schaalschatting (`scalesd`), die wordt toegevoegd aan `texeln_stats_omni_meas2.pkl`.

In [ ]:
yrmbs = streams["scale_bootstrap"].sample(yrm, nb * len(yrm)).reshape(nb, len(yrm)).T
scalebs = np.full(nb, np.nan)
for ib in range(nb):
    wlmbs = wlm[collect(yr, yrmbs[:, ib]) - 1]
    Xb = -np.sort(-wlmbs)
    es = FitGW_iHilli(
        Xb[: int(l0m.max()) + 20],
        WL.p,
        N=Nm,
        r11=1,
        fixedpar={"theta0": esGW["tailindex"][k], "theta0Std": 0.0},
        l0=l0m,
        sigma=np.inf,
        metadata=md,
    )
    scalebs[ib] = es["scale"][k]
save_pkl(
    paths.out(f"{station}_stats_omni_meas2.pkl"),
    mum=mum,
    mum0=mum0,
    freq=WL.freq,
    qmum=qmum,
    qmsd=qmsd,
    scalesd=r_sd(scalebs),
)
print(f"standaardafwijking van de schaal: {r_sd(scalebs):.4f}")